# Stage 09 — ABC-XYZ-FSN Classification + K-Means
**Dashboard page:** Inventory Analysis
**Tabs:** Classification · Inventory · Movements · Spare Parts EDA · Planning Table

**ABC:** A=top 80% cumulative value, B=next 15%, C=rest
**XYZ:** X=CV<0.5, Y=0.5–1.0, Z≥1.0
**FSN:** F=last issue ≤3m, S=3–12m, N>12m

In [ ]:
import sys, warnings, re
from pathlib import Path
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

INTERIM   = PROJECT_ROOT / "data" / "interim"
PROCESSED = PROJECT_ROOT / "data" / "processed"
OUTPUTS   = PROJECT_ROOT / "data" / "outputs"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:,.2f}".format)
plt.rcParams.update({
    "figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "font.size": 10,
})
PALETTE = ["#4361EE","#EF4444","#2CC56F","#F59E0B","#A855F7",
           "#64748B","#06B6D4","#F97316","#10B981","#8B5CF6"]
STATUS_COLORS = {"stockout":"#EF4444","critical":"#F97316",
                 "low":"#F59E0B","ok":"#2CC56F","excess":"#4361EE"}
TIER_COLORS   = {"critical":"#EF4444","managed":"#F97316",
                 "watch":"#F59E0B","rationalise":"#94A3B8"}

def load(name, base=None):
    if base:
        p = Path(base) / name
        if p.exists(): return pd.read_parquet(p)
    for b in [INTERIM, PROCESSED, OUTPUTS]:
        p = b / name
        if p.exists(): return pd.read_parquet(p)
    raise FileNotFoundError(f"{name} not found")

def fmt_lkr(v):
    if abs(v) >= 1e9: return f"LKR {v/1e9:.1f}B"
    if abs(v) >= 1e6: return f"LKR {v/1e6:.1f}M"
    return f"LKR {v:,.0f}"


In [ ]:
clf = load("abc_xyz_fsn.parquet")
pol = load("inventory_policy.parquet")
mv  = load("monthly_demand.parquet")
feat= load("spare_parts_features.parquet")

print(f"ABC-XYZ-FSN : {len(clf):,} SKUs")
print(f"Policy table: {len(pol):,} SKUs")
print(f"Movements   : {len(mv):,} SKU-month rows")


## Classification Tab — ABC / XYZ / FSN distributions

In [ ]:
fig,axes = plt.subplots(2,2,figsize=(14,10))

# Pareto curve
val_s  = clf.sort_values("total_issue_value_lkr",ascending=False)["total_issue_value_lkr"]
cumval = val_s.cumsum()/val_s.sum()*100
sku_pct= np.arange(1,len(cumval)+1)/len(cumval)*100
axes[0,0].plot(sku_pct,cumval.values,color=PALETTE[0],lw=2.5,zorder=3)
axes[0,0].axhline(80,color=PALETTE[1],ls="--",alpha=0.8,label="80% -> A class")
axes[0,0].axhline(95,color=PALETTE[3],ls="--",alpha=0.8,label="95% -> B class")
a_cut = float(sku_pct[(cumval.values<80)].max()) if (cumval.values<80).any() else 0.0
axes[0,0].axvline(a_cut,color=PALETTE[1],ls=":",alpha=0.5)
axes[0,0].set_title(f"Pareto Curve — A-class: {a_cut:.1f}% of SKUs = 80% of value")
axes[0,0].set_xlabel("Cumulative % SKUs"); axes[0,0].set_ylabel("Cumulative % value")
axes[0,0].legend(fontsize=8)

# ABC/XYZ/FSN bars
for i,(dim,order) in enumerate([("abc",["A","B","C"]),("xyz",["X","Y","Z"]),("fsn",["F","S","N"])]):
    ax = axes[0,1] if i==0 else axes[1,0] if i==1 else axes[1,1]
    cnt = clf[dim].value_counts().reindex(order,fill_value=0)
    cnt.plot(kind="bar",ax=ax,color=PALETTE[:3],edgecolor="white")
    ax.set_title(f"{dim.upper()} Distribution"); ax.set_ylabel("SKUs")
    ax.tick_params(axis="x",rotation=0)
    for bar,val in zip(ax.patches,cnt.values):
        ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+10,str(val),ha="center",fontsize=9)
plt.tight_layout(); plt.show()


In [ ]:
# ABC x XYZ heatmap + Policy tier
cross = clf.groupby(["abc","xyz"]).size().unstack(fill_value=0).reindex(
    index=["A","B","C"],columns=["X","Y","Z"],fill_value=0)
tier_cnt = pol["policy_tier"].value_counts() if "policy_tier" in pol.columns else pd.Series()

fig,axes = plt.subplots(1,2,figsize=(13,4))
sns.heatmap(cross,annot=True,fmt="d",cmap="Blues",ax=axes[0],linewidths=0.5,linecolor="white",
            cbar_kws={"label":"SKU count"})
axes[0].set_title("ABC x XYZ Matrix")

if len(tier_cnt):
    order = ["critical","managed","watch","rationalise"]
    tc = tier_cnt.reindex([x for x in order if x in tier_cnt.index],fill_value=0)
    tc.plot(kind="bar",ax=axes[1],color=[TIER_COLORS.get(k,"#94A3B8") for k in tc.index],edgecolor="white")
    axes[1].set_title("Policy Tier Distribution
(critical=A+F, managed=A/B+S, watch=C+F, rationalise=rest)")
    axes[1].set_ylabel("SKUs"); axes[1].tick_params(axis="x",rotation=0)
plt.tight_layout(); plt.show()


## Inventory Tab — Stock Status

In [ ]:
sc = pol["stock_status"].value_counts() if "stock_status" in pol.columns else pd.Series()
if len(sc):
    order = ["stockout","critical","low","ok","excess"]
    sc = sc.reindex([x for x in order if x in sc.index],fill_value=0)
    colors = [STATUS_COLORS.get(s,"#94A3B8") for s in sc.index]
    fig,axes = plt.subplots(1,2,figsize=(13,4))
    sc.plot(kind="bar",ax=axes[0],color=colors,edgecolor="white")
    axes[0].set_title("Stock Status Distribution
(Critical=coverage<1m, Low=1-3m, OK=3-6m, Excess>6m)")
    axes[0].tick_params(axis="x",rotation=0)
    axes[1].pie(sc.values,labels=sc.index,colors=colors,autopct="%1.0f%%",startangle=90)
    axes[1].set_title("Stock Status Share"); plt.tight_layout(); plt.show()
    print(sc.to_string())


## Movements Tab — Monthly Issue Demand

In [ ]:
month_col = "year_month_str" if "year_month_str" in mv.columns else None
issue_col = "issue_qty"      if "issue_qty"      in mv.columns else None
if month_col and issue_col:
    agg = mv.groupby(month_col).agg(total_qty=(issue_col,"sum"),sku_count=("material_9","nunique"))
    fig,axes = plt.subplots(2,1,figsize=(13,7),sharex=True)
    agg["total_qty"].plot(ax=axes[0],color=PALETTE[0],lw=2)
    axes[0].set_title("Total Monthly Issue Quantity — All SKUs")
    axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f"{int(x):,}"))
    agg["sku_count"].plot(ax=axes[1],color=PALETTE[2],lw=2)
    axes[1].set_title("Active SKU Count per Month (at least 1 issue)")
    plt.xticks(rotation=45,ha="right"); plt.tight_layout(); plt.show()
    print(f"Total issues : {agg['total_qty'].sum():,.0f} units across {len(agg)} months")


## Planning Table (Module 4)

In [ ]:
try:
    pt = load("m4_planning_table.parquet")
    print(f"Planning table: {len(pt):,} SKUs | cols: {pt.columns.tolist()}")
    print(pt.head(10).to_string())
    # Urgency distribution
    urg_col = next((c for c in ["order_urgency","urgency"] if c in pt.columns),None)
    if urg_col:
        urg = pt[urg_col].value_counts()
        fig,ax = plt.subplots(figsize=(8,4))
        urg.plot(kind="bar",ax=ax,color=[PALETTE[i%len(PALETTE)] for i in range(len(urg))],edgecolor="white")
        ax.set_title("Planning Table: Order Urgency Distribution"); ax.set_ylabel("SKUs")
        ax.tick_params(axis="x",rotation=30); plt.tight_layout(); plt.show()
except FileNotFoundError:
    print("M4 planning table not yet computed.")
